# Scope Decision: K and DEF Excluded from Full Modeling

This is not a modeling notebook — it's a short, documented engineering decision, made before Phase 4 starts, so a reviewer sees it as a deliberate call backed by evidence rather than an unexplained gap in position coverage.

**Decision: no walk-forward-trained XGBoost model, no RFECV-tuned feature set, and no SHAP explainer will be built for Kicker or Team Defense.** Phase 4 onward (model training, SHAP interpretability, and the trade-fairness engine's KVS scoring) is rescoped to **QB, RB, WR, TE only**.

## Reason 1: Neither position is ever actually traded in this league

Checked directly against this league's real transaction history, not assumed: every trade transaction across every season in the Sleeper league chain (2024–2026) was pulled and checked for any K or DEF player among the assets involved.

**Result: zero trades, in this league's entire recorded history, have ever involved a K or a DEF.**

The entire point of Stage 1's Keeper Value Score is to price *trade* value. Building full modeling infrastructure — walk-forward tuning, SHAP explainability, a maintained feature pipeline — for two positions that structurally never enter a trade in this league spends real engineering effort on a capability the trade-fairness engine will never actually call.

## Reason 2: DEF's own evidence argues against it, independent of the trading question

From the naive-baseline reality check in `05_feature_selection.ipynb`: an untuned XGBoost model trained on DEF's (thin, 3-candidate) feature set came back with a **worse** Spearman rank correlation than the trivial "assume next season repeats this season" baseline:

- Naive Spearman: **0.307**
- XGBoost Spearman: **0.188**

That's not "a small lift" or "a low ceiling" — it's the model actively *ranking worse* than doing nothing. Combined with DEF being a team-unit position with no individual-player-shaped data to draw on (only `scarcity_z`, `position_std_vorp_that_season`, and `vorp_delta_yoy` were even structurally available as candidates) and defense's well-known real-world year-to-year volatility (turnover luck, scheme and personnel churn), this is real evidence the position isn't well-modeled by what's currently available — not just an unfavorable trading use case.

## Reason 3: K's result was also weak, and its one apparent signal didn't hold up

K's untuned model barely cleared the naive baseline on Spearman (0.366 vs. 0.347 naive — a gain of 0.019, the smallest positive margin of any position that did beat naive). The one feature that looked like a real, meaningful signal in K's gain-importance chart — `draft_tier_Round 2-3` — was checked directly against how many real kickers that draft tier actually represents, rather than trusted at face value:

**Only 5 distinct kickers in the entire dataset (27 of 733 K-season rows) were ever drafted in Round 2–3.** That's one idiosyncratic subgroup producing an outsized importance score, not a generalizable pattern — confirmed by contrast against the same check run on TE's version of the same feature, which held up (97 distinct players, 525 rows).

## The evidence side by side: naive baseline vs. untuned XGBoost, all 6 positions

Pulled directly from `05_feature_selection.ipynb`'s naive-baseline comparison (same walk-forward folds, same test rows, same metrics for both) — the full picture, not just the two positions being excluded, so the QB/RB/WR/TE numbers that justify keeping those four are visible right alongside the K/DEF numbers that justify dropping them.

In [1]:
import pandas as pd

# Sourced directly from 05_feature_selection.ipynb's naive-vs-XGBoost comparison cell
# (same walk-forward folds, same test rows, same MAE/Spearman definitions).
comparison_df = pd.DataFrame([
    {"position": "QB",  "naive_mae": 72.48, "xgboost_mae": 68.13, "mae_improvement_pct": "6.0%",  "naive_spearman": 0.666, "xgboost_spearman": 0.694, "spearman_delta": 0.029,  "verdict": "Marginal lift"},
    {"position": "RB",  "naive_mae": 47.97, "xgboost_mae": 47.52, "mae_improvement_pct": "0.9%",  "naive_spearman": 0.683, "xgboost_spearman": 0.691, "spearman_delta": 0.008,  "verdict": "Marginal lift"},
    {"position": "WR",  "naive_mae": 39.34, "xgboost_mae": 36.09, "mae_improvement_pct": "8.2%",  "naive_spearman": 0.730, "xgboost_spearman": 0.753, "spearman_delta": 0.023,  "verdict": "Marginal lift"},
    {"position": "TE",  "naive_mae": 27.28, "xgboost_mae": 26.14, "mae_improvement_pct": "4.2%",  "naive_spearman": 0.691, "xgboost_spearman": 0.699, "spearman_delta": 0.008,  "verdict": "Marginal lift"},
    {"position": "K",   "naive_mae": 36.86, "xgboost_mae": 33.46, "mae_improvement_pct": "9.2%",  "naive_spearman": 0.347, "xgboost_spearman": 0.366, "spearman_delta": 0.019,  "verdict": "Marginal lift -- EXCLUDED (reasons above)"},
    {"position": "DEF", "naive_mae": 29.81, "xgboost_mae": 26.27, "mae_improvement_pct": "11.9%", "naive_spearman": 0.307, "xgboost_spearman": 0.188, "spearman_delta": -0.119, "verdict": "WORSE ranker than naive -- EXCLUDED (reasons above)"},
])

pd.set_option("display.max_colwidth", None)
print(comparison_df.to_string(index=False))

position  naive_mae  xgboost_mae mae_improvement_pct  naive_spearman  xgboost_spearman  spearman_delta                                             verdict
      QB      72.48        68.13                6.0%           0.666             0.694           0.029                                       Marginal lift
      RB      47.97        47.52                0.9%           0.683             0.691           0.008                                       Marginal lift
      WR      39.34        36.09                8.2%           0.730             0.753           0.023                                       Marginal lift
      TE      27.28        26.14                4.2%           0.691             0.699           0.008                                       Marginal lift
       K      36.86        33.46                9.2%           0.347             0.366           0.019           Marginal lift -- EXCLUDED (reasons above)
     DEF      29.81        26.27               11.9%           0.307  

**Note on QB/RB/WR/TE:** none of the four shows a *large* lift either (all "marginal" by the same threshold used in `05_feature_selection.ipynb`) — that's a separate, already-documented finding for Phase 4 to address with real hyperparameter tuning. What matters here is the *relative* picture: all four beat naive on both metrics, in the same direction, consistently. K barely does. DEF doesn't.

## What replaces full modeling for K/DEF going forward

Wherever K or DEF value needs to be referenced elsewhere in the pipeline (roster overviews, keeper-cost context, a trade that unexpectedly does include one of them) — a **simple, explicitly-labeled non-ML heuristic**, not a model:

- **Current-season raw VORP** (already computed and sitting in `vorp_labels.parquet` for every K/DEF player-season — no new computation needed), or
- **A short rolling average** (e.g. trailing 2–3 seasons of VORP) if a slightly smoothed number is preferred over a single season's noise.

Either way, this heuristic is **never presented with the same confidence or precision as the four real models** — no SHAP explanation, no fold-validated error bars, no claim that it's predictive of next season beyond "this is what recently happened." If a K or DEF ever *does* show up in a real trade proposal, Stage 2's fairness engine uses this heuristic and says so plainly, rather than quietly running it through infrastructure that was never validated for these two positions.